# 00 — VitalDB Quick Exploration (Pre-Phase 3)

Purpose: validate VitalAgent design assumptions against real VitalDB metadata.

5 analyses:
1. Total case loading
2. Surgery type distribution (`department` / `optype` / `opname`)
3. Demographics (age, sex, weight, height, BMI, ASA)
4. Surgery duration (from `opstart`/`opend`)
5. Modality availability matrix (priority tracks)

Companion script: `docs/notebooks/00_vitaldb_quick_exploration.py`
Findings: `docs/findings/pre_phase3_findings.md`


In [ ]:
import pandas as pd
import numpy as np
import vitaldb
print("vitaldb:", getattr(vitaldb, "__version__", "1.6.0"))


## Load metadata (cached)

The in-package `vitaldb.load_clinical_data()` returned 0 rows in this environment (probably needs login). The public CSV endpoint is the operable path.

In [ ]:
CASES_URL = "https://api.vitaldb.net/cases"
TRKS_URL = "https://api.vitaldb.net/trks"

import os
CACHE = "_cache"
os.makedirs(CACHE, exist_ok=True)

cases_path = os.path.join(CACHE, "cases.csv")
trks_path = os.path.join(CACHE, "trks.csv")

if os.path.exists(cases_path):
    cases = pd.read_csv(cases_path)
else:
    cases = pd.read_csv(CASES_URL); cases.to_csv(cases_path, index=False)

if os.path.exists(trks_path):
    trks = pd.read_csv(trks_path)
else:
    trks = pd.read_csv(TRKS_URL); trks.to_csv(trks_path, index=False)

print("cases:", len(cases), "cols:", cases.shape[1])
print("trks:", len(trks), "unique tracks:", trks["tname"].nunique())


## 1. Total cases

In [ ]:
print("n_cases:", len(cases))
print("columns:", list(cases.columns))
cases.head(3)


## 2. Surgery type distribution

In [ ]:
print("department:"); print(cases["department"].value_counts())
print()
print("optype top10:"); print(cases["optype"].value_counts().head(10))
print()
print("approach:"); print(cases["approach"].value_counts())
print()
print("opname unique:", cases["opname"].nunique())
print("opname top10:"); print(cases["opname"].value_counts().head(10))


## 3. Demographics

In [ ]:
for col in ["age", "weight", "height", "bmi", "asa"]:
    s = cases[col].dropna()
    print(f"{col}: n={len(s)} missing={cases[col].isna().sum()} "
          f"mean={s.mean():.1f}±{s.std():.1f} "
          f"p5/50/95={s.quantile(0.05):.1f}/{s.quantile(0.5):.1f}/{s.quantile(0.95):.1f} "
          f"min/max={s.min():.1f}/{s.max():.1f}")
print()
print("sex:"); print(cases["sex"].value_counts())


## 4. Surgery duration

In [ ]:
dur_min = (cases["opend"] - cases["opstart"]) / 60.0
d = dur_min.dropna()
print(f"n={len(d)} mean={d.mean():.1f}±{d.std():.1f} min")
print(f"percentiles: p5={d.quantile(0.05):.1f} p25={d.quantile(0.25):.1f} "
      f"p50={d.quantile(0.5):.1f} p75={d.quantile(0.75):.1f} p95={d.quantile(0.95):.1f}")
print(f"range: {d.min():.1f} – {d.max():.1f} min")
print()
bins = {"<30": (d<30).sum(), "30-60": ((d>=30)&(d<60)).sum(),
        "60-120": ((d>=60)&(d<120)).sum(), "120-240": ((d>=120)&(d<240)).sum(),
        ">=240": (d>=240).sum()}
print("duration bins:", bins)
print(f"<30min exclusion drops: {bins['<30']} cases -> remaining {len(d) - bins['<30']}")


## 5. Modality availability matrix (priority tracks)

In [ ]:
priority = [
    "SNUADC/ART", "SNUADC/PLETH", "SNUADC/ECG_II",
    "Solar8000/ART_MBP", "Solar8000/NIBP_MBP",
    "BIS/EEG1_WAV", "Primus/SEVOFLURANE_VOL", "Orchestra/PPF20_CE",
]
per_track = trks.groupby("tname")["caseid"].nunique()
for t in priority:
    n = int(per_track.get(t, 0))
    print(f"{t}: {n} ({100*n/len(cases):.1f}%)")


### ⚠️ `Primus/SEVOFLURANE_VOL` returns 0 — wrong track name.
Actual sevoflurane channels:

In [ ]:
for t in per_track.index:
    if "sevo" in t.lower() or "flur" in t.lower():
        print(f"  {t}: {per_track[t]} cases")


### ABP family (invasive + numeric + auxiliary)

In [ ]:
case_to_tracks = trks.groupby("caseid")["tname"].agg(set)
def has(t):
    return case_to_tracks.apply(lambda x: t in x).reindex(cases["caseid"].values, fill_value=False)

h_art = has("SNUADC/ART")
h_artmbp = has("Solar8000/ART_MBP")
abp_any = h_art | h_artmbp
print(f"ABP invasive (SNUADC/ART): {h_art.mean()*100:.1f}%")
print(f"ABP numeric (Solar8000/ART_MBP): {h_artmbp.mean()*100:.1f}%")
print(f"ABP any: {abp_any.mean()*100:.1f}%")
print(f"ABP absent: {(~abp_any).mean()*100:.1f}% ({(~abp_any).sum()} cases)")
print()
print("ABP absent by department:")
for d_name in cases["department"].unique():
    mask = cases["department"].values == d_name
    if mask.sum() >= 50:
        absent = (~abp_any.values[mask]).mean() * 100
        print(f"  {d_name}: {absent:.1f}% absent ({mask.sum()} total)")


### Top 20 most-available tracks

In [ ]:
per_track.sort_values(ascending=False).head(20)

## 6. Bottom line

See `docs/findings/pre_phase3_findings.md` for the full reconciliation against project_brief assumptions.

Key results:
- **6,388 cases — exact match with brief §4** ✅
- **Department 4-bucket = project taxonomy** ✅
- **ABP absent in 41.7% of cases** (department-stratified: General 51.9% vs Thoracic 2.5%) — strong modality-agnostic story but **must be reported stratified**
- **Sevoflurane track name in brief is wrong** — `Primus/SEVOFLURANE_VOL` → `Primus/EXP_SEVO`
- **<30 min exclusion drops 442 cases → 5,946 remaining** (inside the 5,800–6,000 band)
- Pediatric cases (min age 0.3 yr) and ASA=6 (donor) cases are present — clinician decision needed.

Next: apply the 6 small patches to brief / master_plan / plan_1.{1,2,3,5} and enter Phase 3.